In [30]:
import sys; sys.path.insert(0, '../src')
import sqlite3
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit

from bradley_terry import filter_complete, build_design_matrix, predict
from ordinal import predict_expected_margin
from regime_decay import compute_weights_regime
from team_prior import fit_with_new_player_prior

# ── 1. Load the database ───────────────────────────────────────────────
conn = sqlite3.connect('../data/processed/idv.db')
matches = pd.read_sql("SELECT * FROM matches ORDER BY date", conn)
conn.close()
d = filter_complete(matches).sort_values('date').reset_index(drop=True)

# ── 2. Choose your hyperparameters ─────────────────────────────────────
TAU_PRE    = 136      # half-life (days) for matches BEFORE shift_date
TAU_POST   = 136       # half-life (days) for matches AFTER shift_date
ALPHA      = 1        # 0–1; multiplicative discount on pre-shift matches
SHIFT_DATE = '2023-01-01'
L2_LAMBDA  = 1.0         # ridge strength
THRESHOLD  = 5           # players with <THRESHOLD training matches get team-mean prior
MODEL      = 'ordinal'   # 'ordinal' or 'linear'

# ── 3. Choose what to test on ──────────────────────────────────────────
TEST_TIERS = ['IVL', 'IJL', 'COA', 'IVS', 'IVT', 'IVC']           # restrict TEST set to these tiers
TRAIN_TIERS = ['IVL', 'IJL', 'COA', 'IVS', 'IVT', 'IVC']
N_SPLITS   = 5

# ── 4. Run 5-fold temporal CV ──────────────────────────────────────────
train_mask = d['tournament_tier'].isin(TRAIN_TIERS).to_numpy()
test_mask  = d['tournament_tier'].isin(TEST_TIERS).to_numpy()
splits = [
    (tr[train_mask[tr]], te[test_mask[te]])
    for tr, te in TimeSeriesSplit(n_splits=N_SPLITS).split(d)
]

for TAU in [200, 225, 250, 275]:
    print("Now testing on TAU = ", TAU, "\n")
    TAU_PRE = TAU
    TAU_POST = TAU
    rmses, nulls = [], []
    for fold, (tr, te) in enumerate(splits, 1):
        train, test = d.iloc[tr], d.iloc[te]
        if len(test) == 0:
            continue
    
        weights = compute_weights_regime(
            train['date'], train['date'].max(),
            tau_post=TAU_POST, tau_pre=TAU_PRE,
            shift_date=SHIFT_DATE, alpha=ALPHA,
        )
    
        result = fit_with_new_player_prior(
            train, model=MODEL, l2_lambda=L2_LAMBDA,
            threshold=THRESHOLD, weights=weights,
        )
        if MODEL == 'ordinal':
            beta, idx, res = result
            yhat = predict_expected_margin(test, res)
        else:
            beta, idx = result
            X_te, _   = build_design_matrix(test, idx)
            yhat      = predict(X_te, beta)
    
        y      = (test['n_escaped'] - 2).to_numpy(float)
        rmse   = float(np.sqrt(np.mean((y - yhat) ** 2)))
        null   = float(np.sqrt(np.mean(y ** 2)))
        r2     = 1 - rmse ** 2 / null ** 2
        print(f"  fold {fold}: n={len(test):>5}  RMSE={rmse:.5f}  R²={r2*100:+.2f}%")
        rmses.append(rmse); nulls.append(null)

    # Pooled R²
    pooled_r2 = 1 - np.mean([r**2 for r in rmses]) / np.mean([n**2 for n in nulls])
    print(f"\nPooled R²: {pooled_r2*100:.2f}%")

  filter_complete: dropped 102 incomplete row(s)
Now testing on TAU =  200 

  fold 1: n= 2598  RMSE=1.18643  R²=+4.42%
  fold 2: n= 2598  RMSE=1.13600  R²=+5.05%
  fold 3: n= 2598  RMSE=1.13828  R²=+5.87%
  fold 4: n= 2598  RMSE=1.11653  R²=+14.77%
  fold 5: n= 2598  RMSE=1.03098  R²=+14.43%

Pooled R²: 8.82%
Now testing on TAU =  225 

  fold 1: n= 2598  RMSE=1.18605  R²=+4.48%
  fold 2: n= 2598  RMSE=1.13548  R²=+5.14%
  fold 3: n= 2598  RMSE=1.13846  R²=+5.84%
  fold 4: n= 2598  RMSE=1.11703  R²=+14.69%
  fold 5: n= 2598  RMSE=1.03078  R²=+14.46%

Pooled R²: 8.83%
Now testing on TAU =  250 

  fold 1: n= 2598  RMSE=1.18576  R²=+4.52%
  fold 2: n= 2598  RMSE=1.13508  R²=+5.20%
  fold 3: n= 2598  RMSE=1.13872  R²=+5.80%
  fold 4: n= 2598  RMSE=1.11748  R²=+14.62%
  fold 5: n= 2598  RMSE=1.03073  R²=+14.47%

Pooled R²: 8.83%
Now testing on TAU =  275 

  fold 1: n= 2598  RMSE=1.18556  R²=+4.56%
  fold 2: n= 2598  RMSE=1.13479  R²=+5.25%
  fold 3: n= 2598  RMSE=1.13901  R²=+5.75%
  fol

In [38]:
import sys; sys.path.insert(0, '../src')
import sqlite3
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit

from bradley_terry import filter_complete, build_design_matrix, predict
from ordinal import predict_expected_margin
from regime_decay import compute_weights_regime
from team_prior import fit_with_new_player_prior

# ── 1. Load the database ───────────────────────────────────────────────
conn = sqlite3.connect('../data/processed/idv.db')
matches = pd.read_sql("SELECT * FROM matches ORDER BY date", conn)
conn.close()
d = filter_complete(matches).sort_values('date').reset_index(drop=True)

# ── 2. Choose your hyperparameters ─────────────────────────────────────
TAU_PRE    = 136      # half-life (days) for matches BEFORE shift_date
TAU_POST   = 136       # half-life (days) for matches AFTER shift_date
ALPHA      = 0.6        # 0–1; multiplicative discount on pre-shift matches
SHIFT_DATE = '2023-01-01'
L2_LAMBDA  = 1.0         # ridge strength
THRESHOLD  = 5           # players with <THRESHOLD training matches get team-mean prior
MODEL      = 'ordinal'   # 'ordinal' or 'linear'

# ── 3. Choose what to test on ──────────────────────────────────────────
TEST_TIERS = ['IVL']           # restrict TEST set to these tiers
TRAIN_TIERS = ['IVL']
N_SPLITS   = 5

# ── 4. Run 5-fold temporal CV ──────────────────────────────────────────
train_mask = d['tournament_tier'].isin(TRAIN_TIERS).to_numpy()
test_mask  = d['tournament_tier'].isin(TEST_TIERS).to_numpy()
splits = [
    (tr[train_mask[tr]], te[test_mask[te]])
    for tr, te in TimeSeriesSplit(n_splits=N_SPLITS).split(d)
]

for TAU in [60, 100, 150, 200, 250, 300]:
    print("Now testing on TAU = ", TAU, "\n")
    TAU_PRE = TAU + 50
    TAU_POST = TAU
    rmses, nulls = [], []
    for fold, (tr, te) in enumerate(splits, 1):
        train, test = d.iloc[tr], d.iloc[te]
        if len(test) == 0:
            continue
    
        weights = compute_weights_regime(
            train['date'], train['date'].max(),
            tau_post=TAU_POST, tau_pre=TAU_PRE,
            shift_date=SHIFT_DATE, alpha=ALPHA,
        )
    
        result = fit_with_new_player_prior(
            train, model=MODEL, l2_lambda=L2_LAMBDA,
            threshold=THRESHOLD, weights=weights,
        )
        if MODEL == 'ordinal':
            beta, idx, res = result
            yhat = predict_expected_margin(test, res)
        else:
            beta, idx = result
            X_te, _   = build_design_matrix(test, idx)
            yhat      = predict(X_te, beta)
    
        y      = (test['n_escaped'] - 2).to_numpy(float)
        rmse   = float(np.sqrt(np.mean((y - yhat) ** 2)))
        null   = float(np.sqrt(np.mean(y ** 2)))
        r2     = 1 - rmse ** 2 / null ** 2
        print(f"  fold {fold}: n={len(test):>5}  RMSE={rmse:.5f}  R²={r2*100:+.2f}%")
        rmses.append(rmse); nulls.append(null)

    # Pooled R²
    pooled_r2 = 1 - np.mean([r**2 for r in rmses]) / np.mean([n**2 for n in nulls])
    print(f"\nPooled R²: {pooled_r2*100:.2f}%")

  filter_complete: dropped 102 incomplete row(s)
Now testing on TAU =  60 

  fold 1: n= 1095  RMSE=1.14167  R²=+2.24%
  fold 2: n=  947  RMSE=1.06182  R²=+4.16%
  fold 3: n= 1138  RMSE=1.10772  R²=+0.19%
  fold 4: n= 1082  RMSE=1.06602  R²=+7.69%
  fold 5: n= 1125  RMSE=1.05338  R²=+6.14%

Pooled R²: 4.04%
Now testing on TAU =  100 

  fold 1: n= 1095  RMSE=1.13953  R²=+2.61%
  fold 2: n=  947  RMSE=1.05991  R²=+4.50%
  fold 3: n= 1138  RMSE=1.10878  R²=-0.00%
  fold 4: n= 1082  RMSE=1.05979  R²=+8.77%
  fold 5: n= 1125  RMSE=1.04593  R²=+7.47%

Pooled R²: 4.61%
Now testing on TAU =  150 

  fold 1: n= 1095  RMSE=1.13836  R²=+2.81%
  fold 2: n=  947  RMSE=1.05879  R²=+4.70%
  fold 3: n= 1138  RMSE=1.11035  R²=-0.29%
  fold 4: n= 1082  RMSE=1.06053  R²=+8.64%
  fold 5: n= 1125  RMSE=1.04110  R²=+8.32%

Pooled R²: 4.78%
Now testing on TAU =  200 

  fold 1: n= 1095  RMSE=1.13786  R²=+2.90%
  fold 2: n=  947  RMSE=1.05827  R²=+4.80%
  fold 3: n= 1138  RMSE=1.11190  R²=-0.57%
  fold 4: n=

In [ ]:
# good when we include everything in testing
# IVL is the only thing that's contributing to the weird fold 3 data

In [39]:
import sys; sys.path.insert(0, '../src')
import sqlite3
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit

from bradley_terry import filter_complete, build_design_matrix, predict
from ordinal import predict_expected_margin
from regime_decay import compute_weights_regime
from team_prior import fit_with_new_player_prior

# ── 1. Load the database ───────────────────────────────────────────────
conn = sqlite3.connect('../data/processed/idv.db')
matches = pd.read_sql("SELECT * FROM matches ORDER BY date", conn)
conn.close()
d = filter_complete(matches).sort_values('date').reset_index(drop=True)

# ── 2. Choose your hyperparameters ─────────────────────────────────────
TAU_PRE    = 136      # half-life (days) for matches BEFORE shift_date
TAU_POST   = 136       # half-life (days) for matches AFTER shift_date
ALPHA      = 0.6        # 0–1; multiplicative discount on pre-shift matches
SHIFT_DATE = '2023-01-01'
L2_LAMBDA  = 1.0         # ridge strength
THRESHOLD  = 5           # players with <THRESHOLD training matches get team-mean prior
MODEL      = 'ordinal'   # 'ordinal' or 'linear'

# ── 3. Choose what to test on ──────────────────────────────────────────
TEST_TIERS = ['COA','IVL', 'IJL']           # restrict TEST set to these tiers
TRAIN_TIERS = ['COA','IVL', 'IJL']   
N_SPLITS   = 5

# ── 4. Run 5-fold temporal CV ──────────────────────────────────────────
train_mask = d['tournament_tier'].isin(TRAIN_TIERS).to_numpy()
test_mask  = d['tournament_tier'].isin(TEST_TIERS).to_numpy()
splits = [
    (tr[train_mask[tr]], te[test_mask[te]])
    for tr, te in TimeSeriesSplit(n_splits=N_SPLITS).split(d)
]

for TAU in [60, 100, 150, 200, 250, 300]:
    print("Now testing on TAU = ", TAU, "\n")
    TAU_PRE = TAU + 50
    TAU_POST = TAU
    rmses, nulls = [], []
    for fold, (tr, te) in enumerate(splits, 1):
        train, test = d.iloc[tr], d.iloc[te]
        if len(test) == 0:
            continue
    
        weights = compute_weights_regime(
            train['date'], train['date'].max(),
            tau_post=TAU_POST, tau_pre=TAU_PRE,
            shift_date=SHIFT_DATE, alpha=ALPHA,
        )
    
        result = fit_with_new_player_prior(
            train, model=MODEL, l2_lambda=L2_LAMBDA,
            threshold=THRESHOLD, weights=weights,
        )
        if MODEL == 'ordinal':
            beta, idx, res = result
            yhat = predict_expected_margin(test, res)
        else:
            beta, idx = result
            X_te, _   = build_design_matrix(test, idx)
            yhat      = predict(X_te, beta)
    
        y      = (test['n_escaped'] - 2).to_numpy(float)
        rmse   = float(np.sqrt(np.mean((y - yhat) ** 2)))
        null   = float(np.sqrt(np.mean(y ** 2)))
        r2     = 1 - rmse ** 2 / null ** 2
        print(f"  fold {fold}: n={len(test):>5}  RMSE={rmse:.5f}  R²={r2*100:+.2f}%")
        rmses.append(rmse); nulls.append(null)

    # Pooled R²
    pooled_r2 = 1 - np.mean([r**2 for r in rmses]) / np.mean([n**2 for n in nulls])
    print(f"\nPooled R²: {pooled_r2*100:.2f}%")

  filter_complete: dropped 102 incomplete row(s)
Now testing on TAU =  60 

  fold 1: n= 1846  RMSE=1.17191  R²=+2.79%
  fold 2: n= 2184  RMSE=1.10333  R²=+4.78%
  fold 3: n= 2421  RMSE=1.14928  R²=+3.36%
  fold 4: n= 2353  RMSE=1.09750  R²=+14.94%
  fold 5: n= 2523  RMSE=1.03697  R²=+12.57%

Pooled R²: 7.65%
Now testing on TAU =  100 

  fold 1: n= 1846  RMSE=1.17053  R²=+3.02%
  fold 2: n= 2184  RMSE=1.10216  R²=+4.98%
  fold 3: n= 2421  RMSE=1.14582  R²=+3.94%
  fold 4: n= 2353  RMSE=1.09892  R²=+14.72%
  fold 5: n= 2523  RMSE=1.03249  R²=+13.32%

Pooled R²: 7.94%
Now testing on TAU =  150 

  fold 1: n= 1846  RMSE=1.16985  R²=+3.13%
  fold 2: n= 2184  RMSE=1.10158  R²=+5.08%
  fold 3: n= 2421  RMSE=1.14458  R²=+4.15%
  fold 4: n= 2353  RMSE=1.10238  R²=+14.18%
  fold 5: n= 2523  RMSE=1.02997  R²=+13.75%

Pooled R²: 7.99%
Now testing on TAU =  200 

  fold 1: n= 1846  RMSE=1.16962  R²=+3.17%
  fold 2: n= 2184  RMSE=1.10139  R²=+5.11%
  fold 3: n= 2421  RMSE=1.14418  R²=+4.22%
  fold